In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os
# ================================================
# GOLD LAYER SETUP
# ================================================
SILVER_PATH = "/lakehouse/default/Files/silver"
GOLD_PATH = "/lakehouse/default/Files/gold"

os.makedirs(GOLD_PATH, exist_ok=True)

# Load all Silver tables into memory once
patients    = pd.read_parquet(f"{SILVER_PATH}/patients/part-0.parquet")
encounters  = pd.read_parquet(f"{SILVER_PATH}/encounters/part-0.parquet")
providers   = pd.read_parquet(f"{SILVER_PATH}/providers/part-0.parquet")
claims      = pd.read_parquet(f"{SILVER_PATH}/claims_and_billing/part-0.parquet")
denials     = pd.read_parquet(f"{SILVER_PATH}/denials/part-0.parquet")
diagnoses   = pd.read_parquet(f"{SILVER_PATH}/diagnoses/part-0.parquet")
procedures  = pd.read_parquet(f"{SILVER_PATH}/procedures/part-0.parquet")
medications = pd.read_parquet(f"{SILVER_PATH}/medications/part-0.parquet")
lab_tests   = pd.read_parquet(f"{SILVER_PATH}/lab_tests/part-0.parquet")

print(" All Silver tables loaded into memory!")
print(f"\nPatients:    {len(patients):,}")
print(f"Encounters:  {len(encounters):,}")
print(f"Claims:      {len(claims):,}")
print(f"Denials:     {len(denials):,}")
print(f"Deniagnoses: {len(diagnoses ):,}")
print(f"Procedures:  {len(procedures):,}")
print(f"Medications: {len(medications):,}")
print(f"Lab Tests:   {len(lab_tests):,}")


 All Silver tables loaded into memory!

Patients:    60,000
Encounters:  70,000
Claims:      70,000
Denials:     5,998
Deniagnoses: 70,000
Procedures:  126,021
Medications: 52,500
Lab Tests:   51,565


In [5]:
# ================================================
# GOLD 1: PATIENT SUMMARY
# Business Question: Who are our patients?
# ================================================

# Step 1: Count encounters per patient
encounter_counts = encounters.groupby("patient_id").agg(
    total_visits        = ("encounter_id", "count"),
    first_visit         = ("visit_date", "min"),
    last_visit          = ("visit_date", "max"),
    readmission_count   = ("readmitted_flag", lambda x: (x == "Yes").sum())
).reset_index()

# Step 2: Total spend per patient
spend = claims.groupby("patient_id").agg(
    total_billed        = ("billed_amount", "sum"),
    total_paid          = ("paid_amount", "sum"),
    total_unpaid        = ("unpaid_amount", "sum"),
    total_claims        = ("claim_id", "count"),
    denied_claims       = ("claim_status", lambda x: (x == "Denied").sum())
).reset_index()

# Step 3: Join everything to patients
gold_patient = patients.merge(encounter_counts, on="patient_id", how="left")
gold_patient = gold_patient.merge(spend, on="patient_id", how="left")

# Step 4: Fill nulls for patients with no claims
gold_patient["total_visits"]      = gold_patient["total_visits"].fillna(0)
gold_patient["total_billed"]      = gold_patient["total_billed"].fillna(0)
gold_patient["total_paid"]        = gold_patient["total_paid"].fillna(0)
gold_patient["denied_claims"]     = gold_patient["denied_claims"].fillna(0)

# Step 5: Add Gold timestamp
gold_patient["_gold_load_timestamp"] = datetime.now().isoformat()

# Step 6: Save
os.makedirs(f"{GOLD_PATH}/patient_summary", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(gold_patient),
    f"{GOLD_PATH}/patient_summary/part-0.parquet"
)
print(f" gold/patient_summary — {len(gold_patient):,} rows")
gold_patient[["patient_id", "first_name", "last_name", 
              "age", "insurance_type", "total_visits",
              "total_billed", "total_paid"]].head(5)


 gold/patient_summary — 60,000 rows


,patient_id,first_name,last_name,age,insurance_type,total_visits,total_billed,total_paid
0,PAT000001,Danielle,Johnson,85,UHC,2,3715.77,2021.94
1,PAT000002,Anna,Baldwin,15,AETNA,1,1317.12,880.29
2,PAT000003,James,Jones,4,UHC,1,689.65,474.16
3,PAT000004,Veronica,Bowman,48,BCBS,2,2905.95,1090.13
4,PAT000005,Carl,Gentry,83,BCBS,1,1061.56,1061.56


In [6]:
# ================================================
# GOLD 2: REVENUE SUMMARY
# Business Question: How is our revenue performing?
# ================================================

# Step 1: Revenue by insurance provider
revenue_by_payer = claims.groupby("insurance_provider").agg(
    total_claims        = ("claim_id", "count"),
    total_billed        = ("billed_amount", "sum"),
    total_paid          = ("paid_amount", "sum"),
    total_unpaid        = ("unpaid_amount", "sum"),
    avg_collection_rate = ("collection_rate", "mean"),
    denied_claims       = ("claim_status", lambda x: (x == "Denied").sum())
).reset_index()

# Step 2: Add denial rate
revenue_by_payer["denial_rate"] = (
    revenue_by_payer["denied_claims"] / 
    revenue_by_payer["total_claims"] * 100
).round(2)

# Step 3: Revenue by month
claims["billing_month"] = claims["claim_billing_date"].dt.to_period("M").astype(str)
revenue_by_month = claims.groupby("billing_month").agg(
    total_billed        = ("billed_amount", "sum"),
    total_paid          = ("paid_amount", "sum"),
    total_claims        = ("claim_id", "count")
).reset_index().sort_values("billing_month")

# Step 4: Revenue by department
claims_enc = claims.merge(
    encounters[["encounter_id", "department"]], 
    on="encounter_id", how="left"
)
revenue_by_dept = claims_enc.groupby("department").agg(
    total_billed        = ("billed_amount", "sum"),
    total_paid          = ("paid_amount", "sum"),
    total_claims        = ("claim_id", "count")
).reset_index().sort_values("total_billed", ascending=False)

# Step 5: Save all 3
os.makedirs(f"{GOLD_PATH}/revenue_by_payer", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(revenue_by_payer),
    f"{GOLD_PATH}/revenue_by_payer/part-0.parquet"
)
print(f" gold/revenue_by_payer — {len(revenue_by_payer):,} rows")

os.makedirs(f"{GOLD_PATH}/revenue_by_month", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(revenue_by_month),
    f"{GOLD_PATH}/revenue_by_month/part-0.parquet"
)
print(f" gold/revenue_by_month — {len(revenue_by_month):,} rows")

os.makedirs(f"{GOLD_PATH}/revenue_by_dept", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(revenue_by_dept),
    f"{GOLD_PATH}/revenue_by_dept/part-0.parquet"
)
print(f" gold/revenue_by_dept — {len(revenue_by_dept):,} rows")

# Preview
print("\n Revenue by Payer:")
revenue_by_payer[["insurance_provider", "total_billed", 
                   "total_paid", "denial_rate"]].head()


 gold/revenue_by_payer — 7 rows
 gold/revenue_by_month — 13 rows
 gold/revenue_by_dept — 21 rows

 Revenue by Payer:


,insurance_provider,total_billed,total_paid,denial_rate
0,AETNA,15931881.91,10220394.03,9.65
1,BCBS,15937527.71,10319121.06,9.79
2,CIGNA,16379391.57,10552126.52,10.27
3,HUMANA,15957291.04,10324831.81,9.99
4,MEDICAID,16297387.90,10458464.11,10.90


In [7]:
# ================================================
# GOLD 3: DENIAL ANALYSIS
# Business Question: Why are claims being denied?
# ================================================

# Step 1: Join denials with claims
denial_analysis = denials.merge(
    claims[["claim_id", "insurance_provider", 
            "billed_amount", "patient_id", "encounter_id"]],
    on="claim_id", how="left"
)

# Step 2: Denial summary by reason
denial_by_reason = denial_analysis.groupby(
    ["denial_reason_code", "denial_reason_description"]
).agg(
    total_denials       = ("denial_id", "count"),
    total_denied_amount = ("denied_amount", "sum"),
    avg_denied_amount   = ("denied_amount", "mean"),
    appeals_filed       = ("appeal_filed", lambda x: (x == "Yes").sum()),
    appeals_approved    = ("appeal_status", lambda x: (x == "Approved").sum()),
    appeals_denied      = ("appeal_status", lambda x: (x == "Denied").sum())
).reset_index()

# Step 3: Add appeal success rate
denial_by_reason["appeal_success_rate"] = (
    denial_by_reason["appeals_approved"] /
    denial_by_reason["appeals_filed"].replace(0, 1) * 100
).round(2)

# Step 4: Denial by insurance provider
denial_by_payer = denial_analysis.groupby("insurance_provider").agg(
    total_denials       = ("denial_id", "count"),
    total_denied_amount = ("denied_amount", "sum"),
    appeals_filed       = ("appeal_filed", lambda x: (x == "Yes").sum()),
    appeals_won         = ("final_outcome", lambda x: (x == "Paid").sum())
).reset_index()

# Step 5: Add win rate
denial_by_payer["appeal_win_rate"] = (
    denial_by_payer["appeals_won"] /
    denial_by_payer["appeals_filed"].replace(0, 1) * 100
).round(2)

# Step 6: Denial trend by month
denial_analysis["denial_month"] = pd.to_datetime(
    denial_analysis["denial_date"]
).dt.to_period("M").astype(str)

denial_by_month = denial_analysis.groupby("denial_month").agg(
    total_denials       = ("denial_id", "count"),
    total_denied_amount = ("denied_amount", "sum")
).reset_index().sort_values("denial_month")

# Step 7: Save all 3
os.makedirs(f"{GOLD_PATH}/denial_by_reason", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(denial_by_reason),
    f"{GOLD_PATH}/denial_by_reason/part-0.parquet"
)
print(f" gold/denial_by_reason — {len(denial_by_reason):,} rows")

os.makedirs(f"{GOLD_PATH}/denial_by_payer", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(denial_by_payer),
    f"{GOLD_PATH}/denial_by_payer/part-0.parquet"
)
print(f" gold/denial_by_payer — {len(denial_by_payer):,} rows")

os.makedirs(f"{GOLD_PATH}/denial_by_month", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(denial_by_month),
    f"{GOLD_PATH}/denial_by_month/part-0.parquet"
)
print(f" gold/denial_by_month — {len(denial_by_month):,} rows")

# Preview
print("\n Top Denial Reasons:")
denial_by_reason[["denial_reason_description",
                   "total_denials", "total_denied_amount",
                   "appeal_success_rate"]]\
    .sort_values("total_denials", ascending=False).head()


 gold/denial_by_reason — 14 rows
 gold/denial_by_payer — 7 rows
 gold/denial_by_month — 5 rows

 Top Denial Reasons:


,denial_reason_description,total_denials,total_denied_amount,appeal_success_rate
3,Duplicate claim/service.,472,708584.38,81.40
4,Precertification/authorization/notification ab...,457,766140.06,80.15
6,Claim denied because it was filed after the ti...,437,716287.07,76.71
13,Non-covered charges,437,656460.88,78.99
5,This care may be covered by another payer per ...,436,676421.46,77.83


In [8]:
# ================================================
# GOLD 4: DEPARTMENT PERFORMANCE
# Business Question: Which departments perform best?
# ================================================

# Step 1: Encounters per department
dept_encounters = encounters.groupby("department").agg(
    total_encounters    = ("encounter_id", "count"),
    avg_length_of_stay  = ("length_of_stay", "mean"),
    total_readmissions  = ("readmitted_flag", lambda x: (x == "Yes").sum()),
    unique_patients     = ("patient_id", "nunique")
).reset_index()

# Step 2: Revenue per department
dept_revenue = claims_enc.groupby("department").agg(
    total_billed        = ("billed_amount", "sum"),
    total_paid          = ("paid_amount", "sum"),
    avg_collection_rate = ("collection_rate", "mean")
).reset_index()

# Step 3: Procedures per department
procedures_enc = procedures.merge(
    encounters[["encounter_id", "department"]],
    on="encounter_id", how="left"
)
dept_procedures = procedures_enc.groupby("department").agg(
    total_procedures    = ("procedure_id", "count"),
    total_procedure_cost = ("procedure_cost", "sum"),
    avg_procedure_cost  = ("procedure_cost", "mean")
).reset_index()

# Step 4: Medications per department
medications_enc = medications.merge(
    encounters[["encounter_id", "department"]],
    on="encounter_id", how="left"
)
dept_medications = medications_enc.groupby("department").agg(
    total_prescriptions = ("medication_id", "count"),
    total_medication_cost = ("cost", "sum"),
    avg_medication_cost = ("cost", "mean")
).reset_index()

# Step 5: Join all together
dept_performance = dept_encounters\
    .merge(dept_revenue,     on="department", how="left")\
    .merge(dept_procedures,  on="department", how="left")\
    .merge(dept_medications, on="department", how="left")

# Step 6: Add readmission rate
dept_performance["readmission_rate"] = (
    dept_performance["total_readmissions"] /
    dept_performance["total_encounters"] * 100
).round(2)

# Step 7: Round floats
dept_performance["avg_length_of_stay"]   = dept_performance["avg_length_of_stay"].round(2)
dept_performance["avg_collection_rate"]  = dept_performance["avg_collection_rate"].round(2)
dept_performance["avg_procedure_cost"]   = dept_performance["avg_procedure_cost"].round(2)
dept_performance["avg_medication_cost"]  = dept_performance["avg_medication_cost"].round(2)

# Step 8: Gold timestamp
dept_performance["_gold_load_timestamp"] = datetime.now().isoformat()

# Step 9: Save
os.makedirs(f"{GOLD_PATH}/department_performance", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(dept_performance),
    f"{GOLD_PATH}/department_performance/part-0.parquet"
)
print(f" gold/department_performance — {len(dept_performance):,} rows")

# Preview
print("\n Department Performance:")
dept_performance[["department", "total_encounters",
                   "total_billed", "total_paid",
                   "readmission_rate"]]\
    .sort_values("total_billed", ascending=False).head()


 gold/department_performance — 21 rows

 Department Performance:


,department,total_encounters,total_billed,total_paid,readmission_rate
3,Emergency Department,16365,23795126.56,15337917.26,2.91
12,Obstetrics & Gynecology,9242,13526292.87,8767434.70,19.55
8,Infectious Disease,2349,12235308.74,7870565.34,19.20
2,Dermatology,2237,5092226.97,3312538.45,19.80
13,Oncology,2253,3744589.28,2382064.69,19.04


In [10]:
# ================================================
# GOLD 5: READMISSION ANALYSIS
# Business Question: Which patients keep coming back?
# ================================================

# Step 1: Flag readmitted encounters
readmissions = encounters[encounters["readmitted_flag"] == "Yes"].copy()

# Step 2: Readmission by department
readmission_by_dept = readmissions.groupby("department").agg(
    total_readmissions  = ("encounter_id", "count"),
    unique_patients     = ("patient_id", "nunique"),
    avg_length_of_stay  = ("length_of_stay", "mean")
).reset_index().sort_values("total_readmissions", ascending=False)

# Step 3: Readmission by diagnosis
readmission_by_diag = readmissions.merge(
    diagnoses[["encounter_id", "diagnosis_description", "chronic_flag"]],
    on="encounter_id", how="left"
)
readmission_by_diag = readmission_by_diag.groupby(
    ["diagnosis_description", "chronic_flag"]
).agg(
    total_readmissions  = ("encounter_id", "count"),
    unique_patients     = ("patient_id", "nunique")
).reset_index().sort_values("total_readmissions", ascending=False)

# Step 4: High risk patients
# Patients with 3+ visits are high risk
high_risk = encounters.groupby("patient_id").agg(
    total_visits        = ("encounter_id", "count"),
    total_readmissions  = ("readmitted_flag", lambda x: (x == "Yes").sum()),
    departments_visited = ("department", "nunique"),
    last_visit          = ("visit_date", "max")
).reset_index()

# Step 5: Join with patient details
high_risk = high_risk.merge(
    patients[["patient_id", "first_name", "last_name",
              "age", "gender", "insurance_type"]],
    on="patient_id", how="left"
)

# Step 6: Add risk score
# Simple scoring: readmissions + age factor + visit frequency
high_risk["risk_score"] = (
    (high_risk["total_readmissions"] * 3) +
    (high_risk["total_visits"] * 1) +
    (high_risk["age"] // 10)
).round(2)

# Step 7: Flag high risk
high_risk["risk_category"] = pd.cut(
    high_risk["risk_score"],
    bins=[0, 5, 10, 20, float("inf")],
    labels=["Low", "Medium", "High", "Critical"]
)

# Step 8: Sort by risk
high_risk = high_risk.sort_values("risk_score", ascending=False)

# Step 9: Readmission by month
encounters["visit_month"] = pd.to_datetime(
    encounters["visit_date"]
).dt.to_period("M").astype(str)

readmission_by_month = encounters.groupby("visit_month").agg(
    total_encounters    = ("encounter_id", "count"),
    total_readmissions  = ("readmitted_flag", lambda x: (x == "Yes").sum())
).reset_index()

readmission_by_month["readmission_rate"] = (
    readmission_by_month["total_readmissions"] /
    readmission_by_month["total_encounters"] * 100
).round(2)

# Step 10: Save all
os.makedirs(f"{GOLD_PATH}/readmission_by_dept", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(readmission_by_dept),
    f"{GOLD_PATH}/readmission_by_dept/part-0.parquet"
)
print(f" gold/readmission_by_dept   — {len(readmission_by_dept):,} rows")

os.makedirs(f"{GOLD_PATH}/readmission_by_diag", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(readmission_by_diag),
    f"{GOLD_PATH}/readmission_by_diag/part-0.parquet"
)
print(f" gold/readmission_by_diag   — {len(readmission_by_diag):,} rows")

os.makedirs(f"{GOLD_PATH}/high_risk_patients", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(high_risk),
    f"{GOLD_PATH}/high_risk_patients/part-0.parquet"
)
print(f" gold/high_risk_patients    — {len(high_risk):,} rows")

os.makedirs(f"{GOLD_PATH}/readmission_by_month", exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(readmission_by_month),
    f"{GOLD_PATH}/readmission_by_month/part-0.parquet"
)
print(f" gold/readmission_by_month  — {len(readmission_by_month):,} rows")

# Preview top high risk patients
print("\n Top 5 High Risk Patients:")
high_risk[["patient_id", "first_name", "last_name",
           "age", "total_readmissions",
           "risk_score", "risk_category"]].head()

 gold/readmission_by_dept   — 21 rows
 gold/readmission_by_diag   — 63 rows
 gold/high_risk_patients    — 60,000 rows
 gold/readmission_by_month  — 3 rows

 Top 5 High Risk Patients:


,patient_id,first_name,last_name,age,total_readmissions,risk_score,risk_category
58377,PAT058378,Natalie,Olson,90,2,18,High
57243,PAT057244,Daniel,Cobb,88,2,17,High
38686,PAT038687,Alyssa,Summers,86,2,17,High
42599,PAT042600,Phillip,Montgomery,88,2,17,High
51463,PAT051464,Veronica,Thomas,85,2,17,High


In [11]:
#register gold tables in Fabric for Power BI

# ================================================
# REGISTER GOLD TABLES IN LAKEHOUSE
# ================================================

GOLD_PATH = "/lakehouse/default/Files/gold"
TABLES_PATH = "/lakehouse/default/Tables"

gold_tables = [
    "patient_summary",
    "revenue_by_payer",
    "revenue_by_month",
    "revenue_by_dept",
    "denial_by_reason",
    "denial_by_payer",
    "denial_by_month",
    "department_performance",
    "readmission_by_dept",
    "readmission_by_diag",
    "high_risk_patients",
    "readmission_by_month"
]

import shutil

for table in gold_tables:
    src  = f"{GOLD_PATH}/{table}/part-0.parquet"
    dest = f"{TABLES_PATH}/gold_{table}"
    os.makedirs(dest, exist_ok=True)
    shutil.copy2(src, f"{dest}/part-0.parquet")
    print(f" Registered: gold_{table}")

print("\nAll Gold tables registered!")
print("Go check your Lakehouse Tables section!")


 Registered: gold_patient_summary
 Registered: gold_revenue_by_payer
 Registered: gold_revenue_by_month
 Registered: gold_revenue_by_dept
 Registered: gold_denial_by_reason
 Registered: gold_denial_by_payer
 Registered: gold_denial_by_month
 Registered: gold_department_performance
 Registered: gold_readmission_by_dept
 Registered: gold_readmission_by_diag
 Registered: gold_high_risk_patients
 Registered: gold_readmission_by_month

All Gold tables registered!
Go check your Lakehouse Tables section!
